# Distillation variants on one GPU (Colab A100)

Runs the student experiments from `ssl_turn/distill/README.md` without Modal compute. Data comes
prepared from the Modal volumes (`colab.py pull`), so nothing is re-encoded. Every run uses
teacher r025 `eotb5_s2` and d003's data recipe (otoSpeech + voice-agent turns + 146 h People's
Speech pool). Variants:

| run | config | what it tests |
|---|---|---|
| c1 | `configs/c1_aug.json` | control s3m vs. room reverb / mic EQ / band limits on the student's input (`acoustic`) |
| c2 | `configs/c2_vad.json` | Silero VAD per frame as an extra student input (`vad_in`); its control has no pool background (VAD is precomputed on clean pool audio) |
| c3 | `configs/c3_feat.json` | feature distillation on all rows including the pool (`feat`; needs `pull --feats`) |
| c4 | `configs/c4_fc.json` | bottom 4 / 2 layers of the pretrained FastConformer under the head (27M / 15M) |
| all | `--select ood` | checkpoint picked by KD loss on otoSpeech dev + held-out voice-agent turns + held-out pool; `--keep-all` keeps every checkpoint |

Plus the X-ASR-zh-en encoder probe at the end. Needs a **High-RAM A100 runtime** (data sits in
~45 GB of host RAM) and ~90 GB of disk (~130 GB with `--feats`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2; df -h /content | tail -1

## Setup
Add `MODAL_TOKEN_ID` and `MODAL_TOKEN_SECRET` (the teaspotato workspace token) as Colab secrets first.

In [ ]:
import os
from google.colab import userdata
os.environ['MODAL_TOKEN_ID'] = userdata.get('MODAL_TOKEN_ID')
os.environ['MODAL_TOKEN_SECRET'] = userdata.get('MODAL_TOKEN_SECRET')
!git clone -q -b claude/project-thread-3utsco https://github.com/shinohara-rin/turn /content/turn || git -C /content/turn pull -q
%cd /content/turn/ssl_turn/distill
!pip -q install modal onnxruntime-gpu kaldi-native-fbank 'nemo_toolkit[asr]==2.4.0'
!pip -q install --no-deps 'silero-vad==6.2.0'
!python colab.py setup

## Data (prepared outputs from the Modal volumes)

In [ ]:
!python colab.py pull            # add --feats for c3 (FastConformer frames of otoSpeech, voice-agent turns, pool)
!du -sh /content/work /content/bgbench

In [ ]:
!python colab.py vad             # Silero per 80 ms frame for c2 only (CPU, all cores)

## Train
Each run trains its students on shared batches, then writes `runs/RUN/probs.npz` (otoSpeech dev + TB dev). If c4 runs out of GPU memory, add `--batch 32`.

In [ ]:
!python colab.py train c1 configs/c1_aug.json --select ood --keep-all 2>&1 | grep -v Warning

In [ ]:
!python colab.py train c2 configs/c2_vad.json --select ood --keep-all 2>&1 | grep -v Warning

In [ ]:
!python colab.py train c3 configs/c3_feat.json --select ood --keep-all 2>&1 | grep -v Warning

In [ ]:
!python colab.py train c4 configs/c4_fc.json --select ood --keep-all 2>&1 | grep -v -i "warning\|nemo w"

## Evaluate

**eot-bench en** (LiveKit's harness in its own venv; it pins numpy<2). Compare with d002/d003 and
teacher r025 `eotb5_s2`: 612 / 385 ms at 5 / 10% (README tables).

In [ ]:
%cd /content
!git clone -q https://github.com/livekit/eot-bench /content/eot-bench 2>/dev/null; python -m venv /content/eotvenv
!/content/eotvenv/bin/pip -q install -e /content/eot-bench 'pyarrow==18.1.0' scikit-learn
!rm -rf /content/en && cp -r /content/eot-bench/output/livekit__eot-bench-data__validation__min_silence_100ms/en /content/en
%cd /content/turn/ssl_turn/distill

In [ ]:
RUNS = {'c1': 's3m,s3m_ac', 'c2': 's3m_nopoolbg,s3m_vad', 'c3': 's3m_feat', 'c4': 'fc4,fc2'}
for run, names in RUNS.items():
    !python colab.py eot {run} {names} --out /content/{run}_tracks.npz 2>&1 | tail -1
    for v in ('eot_q', 'yld_q'):
        !/content/eotvenv/bin/python /content/turn/eotbench/to_harness.py /content/{run}_tracks.npz /content/en --variant {v} --tag {run} | tail -1
!for d in /content/en/ssl_turn__c*; do /content/eotvenv/bin/eot-harness compute-metrics --predictions $d/predictions.parquet --output-dir $d/metrics > /dev/null; done
!/content/eotvenv/bin/python colab.py eotsum '/content/en/ssl_turn__c*'

**TurnBench dev through the commit policy** (pinned TurnBench evaluator in its own venv). Settings are re-selected per model; compare with the fixed-policy table in the README (teacher r025 EOT 0.977 / 0.069 / 548 ms).

In [ ]:
!python -m venv /content/tbvenv
!/content/tbvenv/bin/pip -q install 'git+https://github.com/SesameAILabs/turnbench@38a6f874322430cb3ca71d8a52aa1e636e88bad8' srt scipy modal
for run, names in RUNS.items():
    for name in names.split(','):
        !/content/tbvenv/bin/python colab.py sweep {run} {name} 2>&1 | grep "rules+model:"

**bgbench** (38 TB dev conversations x 14 background conditions). Compare with d003 s3m: playback 1.1, babble 1.5, near talker 7.2 false INT/min; EOT drops -0.07 / -0.08 / -0.16.

In [ ]:
for run, names in RUNS.items():
    !python colab.py bg {run} {names} 2>&1 | tail -1
args = ' '.join(f'--ssl {run}_{n}=/content/bg/{run}.npz:{n}' for run, names in RUNS.items() for n in names.split(','))
!/content/tbvenv/bin/python /content/turn/bgspeech/bench_score.py --plan /content/bg/plan.json {args} --out /content/bg/report 2>&1 | sed -n '/Headline/,/All conditions/p'

**Checkpoint choice.** `runs/RUN/train.json` has every evaluation (otoSpeech dev, held-out voice-agent
turns, held-out pool). To score another checkpoint, copy `runs/RUN/ckpt/NAME_STEP.pt` to
`runs/RUN/NAMEsSTEP.pt` and run the eot / sweep / bg cells with that name.

In [ ]:
import json
h = json.load(open('/work/runs/c1/train.json'))
for name, rows in h['history'].items():
    print(name, 'best', h['best_steps'][name])
    for r in rows:
        print('  ', r['step'], ' '.join(f"{k} {r[k]:.3f}" for k in ('floor', 'eotb_ho_floor', 'pool_ho_floor') if k in r))

## X-ASR-zh-en encoder probe

Linear probe of end-of-turn vs. hold on eot-bench (the multilingual thread's protocol), our
FastConformer vs. X-ASR's 160 ms streaming encoder. Reference (PR #13): fc_en L17+L9 en 0.950,
es 0.941, ja 0.980, zh 0.896.

In [ ]:
%cd /content
!python /content/turn/ssl_turn/distill/xasr_probe.py extract xasr --langs en,zh,es,ja
!python /content/turn/ssl_turn/distill/xasr_probe.py extract fc_en --langs en,zh,es,ja
!python /content/turn/ssl_turn/distill/xasr_probe.py probe feats_fc_en.npz feats_xasr.npz